# Crisis-fork mining — Colab copy 1 of 3 (seeds 60000..69999)

Parallel fork mining. Plays each game to **natural death (no cap)**, rewinds the
last 15-45 plies, and confirms avoidable "crisis fork" moves with R=500 fresh
rollouts. Each confirmed fork lands in a tiny `logs/mine_<seed>.json` (the policy
move is baked in, so death-game trajectories are NOT needed downstream and stay
local). A background job copies mine files to Drive every 5 min, so a disconnect
loses almost nothing.

**Run all 3 copies on separate Colab runtimes at the same time**, alongside the
M5. Seed ranges are disjoint (M5 50140+, copies 60000/70000/80000+) so results
merge cleanly. This is **copy 1** -> `MyDrive/alphatrain/mine_colab_1/`.

## Drive uploads (`MyDrive/alphatrain/`) — same for all 3 copies
1. `colorlines_mining.tar.gz` — code archive (build locally, repo root; explicit
   file list keeps it ~360 KB — do NOT tar `alphatrain/` wholesale, its `data/`
   holds multi-GB checkpoints):
   ```bash
   tar czf colorlines_mining.tar.gz --exclude='**/__pycache__' \
       alphatrain/*.py alphatrain/scripts/*.py \
       alphatrain/data/feature_value_weights_2y_nb.npz \
       scripts/*.py game/
   ```
2. `pillar3b_epoch_20.pt` — already on Drive from pillar3b training.

Upload both, then Runtime > Run all.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil
DRIVE = '/content/drive/MyDrive/alphatrain'
!cp {DRIVE}/colorlines_mining.tar.gz /content/
!cd /content && tar xzf colorlines_mining.tar.gz
os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/pillar3b_epoch_20.pt',
            '/content/alphatrain/data/pillar3b_epoch_20.pt')
%cd /content
os.makedirs('logs', exist_ok=True)
os.makedirs('alphatrain/data/death_games', exist_ok=True)
OUT = f'{DRIVE}/mine_colab_1'
os.makedirs(OUT, exist_ok=True)
# RESUME: pull any already-mined files back so this run SKIPS them (idempotent
# — safe to re-run after a disconnect; it continues where it left off)
!cp -n {OUT}/mine_*.json logs/ 2>/dev/null
print('resuming with', len([x for x in os.listdir('logs') if x.startswith('mine_')]),
      'already-mined seeds')
# crash-safe: copy new mine files to Drive every 5 min (local logs stay fast)
get_ipython().system_raw(
    "nohup bash -c 'while true; do cp -u logs/mine_*.json " + OUT +
    "/ 2>/dev/null; sleep 300; done' >/dev/null 2>&1 &")
!pip install -q numpy numba scipy
print('crash-safe sync every 5 min ->', OUT)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Mine (≈23h, leaves 1h buffer under the 24h Colab limit)

Plays to natural death, mines, confirms. Watch the `death .. -> mine` lines and
`mine rc=0` counts. Mine files are streaming to Drive in the background.

In [ ]:
%cd /content
# --r-screen 50: verified fork-preserving at 1.42x (vs 100); R=500 confirm gates
#   quality, the screen just flags candidates.
# --workers 10: cuda runs the per-process forwards CONCURRENTLY (unlike MPS, which
#   serializes), so workers parallelize the CPU game logic + overlap the GPU.
#   Single-process (workers 1) leaves the L4 (12 cores) mostly idle. ~10 leaves
#   headroom for the main process + recording; tune to the vCPU count.
# Re-runnable: skips seeds already in logs/ (pulled from Drive), so restart resumes.
!python scripts/overnight_systematic.py \
    --device cuda --batch 256 --r-screen 50 --workers 10 \
    --seed-start 60000 --n-try 10000 --max-seconds 82800 \
    2>&1 | tee -a logs/mining_1.log \
    | grep -E "death |mine rc|HARVEST|DONE|forks:|fork\(s\)|skip"


In [ ]:
# Final flush to Drive (catches the last <5 min not yet synced)
import glob, shutil, os
OUT = '/content/drive/MyDrive/alphatrain/mine_colab_1'
n = 0
for f in glob.glob('logs/mine_*.json'):
    shutil.copy(f, OUT); n += 1
if os.path.exists('logs/mining_1.log'):
    shutil.copy('logs/mining_1.log', OUT)
print(f'flushed {n} mine files to {OUT}')

## After all runs finish (on the M5)

Download every `MyDrive/alphatrain/mine_colab_{1,2,3}/mine_*.json` into the repo's
`logs/` (next to the M5's own mine files), then rebuild the combined corpus:
```bash
PYTHONPATH=. python scripts/build_crisis_corpus_file.py \
    --out alphatrain/data/crisis_corpus_v2.pt
```
`build_crisis_corpus` globs all `logs/mine_*.json`; new files carry the policy
move inline (no death games needed), old M5 files fall back to their death games.